# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [ ]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Perder el modelo exacto que se entrenó: es lo único que necesita formato binario; el resto se puede leer sin deserializarlo. |
| Versión del esquema | **manifest.json** | Leer un manifiesto con otra forma (claves distintas) como si fuera el actual. |
| Orden y nombres de features | **manifest.json** (el código lo contrasta con `FEATURE_NAMES`) | Una predicción incorrecta sin error: con las columnas en otro orden el vector sigue teniendo 11 números. |
| Versión del preprocesado | **manifest.json** (el código tiene `PREPROCESSING_VERSION`) | Aplicar en inferencia una transformación distinta a la del entrenamiento. No es lo mismo que `model_version`. |
| Etiquetas de salida permitidas | **manifest.json** (el código las valida en la salida) | Devolver al consumidor una categoría que no espera, como `unknown`. |
| Versión del modelo | **manifest.json** | No saber qué versión del estimador generó una predicción; se copia a cada salida. |
| Tipo de estimador | **manifest.json** (el código comprueba `predict` y `predict_proba`) | Cargar un objeto de otra clase o sin la interfaz que espera la inferencia. |

Las **reglas** (invariantes de carga, validación de entrada y de salida, política de filas del CLI) viven en el **código** y se prueban con tests: ni el binario ni el manifiesto se validan a sí mismos.

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**: `load_model_bundle()` lanza `FileNotFoundError` antes de deserializar nada y no hay inferencia;
2. aparecen features desconocidas o en un orden distinto: el manifiesto no valida (`feature_names` distinto de `FEATURE_NAMES`, o clave desconocida) y la carga falla con `ValueError` antes de abrir **model.joblib**;
3. el modelo devuelve una etiqueta que no está en **output_labels**: `infer_wine_quality()` lanza `ValueError` y esa predicción no llega al CSV;
4. una fila del CSV no cumple el contrato de entrada: `ValueError` con el número de fila y el `sample_id`, y no se escribe ningún CSV.

In [ ]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",  # ARTIFACT_SCHEMA_VERSION de artifact.py
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1",  # PREPROCESSING_VERSION de la semana 3
    "feature_names": feature_names,
    "output_labels": ["needs_review", "acceptable", "excellent"],  # QualityBand del contrato de salida
    "estimator_type": "RandomForestClassifier",  # clase del estimador de la semana 3
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Carga del bundle: se valida `feature_names` del manifiesto antes de abrir **model.joblib** | `feature_names no coincide con FEATURE_NAMES: los nombres y el orden deben ser exactamente [...]` | No |
| Cambia `preprocessing_version` | Carga del bundle: se valida el manifiesto antes de abrir **model.joblib** | `preprocessing_version 'wine-red-features-v2' no coincide con el preprocesado del código ('wine-red-features-v1')` | No |
| El modelo devuelve `unknown` | Validación de la salida en `infer_wine_quality()` | `El modelo devolvió la etiqueta 'unknown', que no está permitida; etiquetas válidas: [...]` | No: la fila falla y el CLI aborta antes de escribir |
| La petición incluye `ph = 99` | Contrato de entrada (`WineQualityRequest`), al validar la fila del CSV | `fila N (sample_id='...'): ph Input should be less than or equal to 4.5` | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.

### Decisiones

- **Campos obligatorios:** los seis; ninguno tiene valor por defecto, porque un valor por defecto escondería una incompatibilidad.
- **Campos desconocidos en el manifiesto:** no se aceptan (`extra="forbid"`).
- **Orden de validación:** primero el manifiesto y después, solo si es válido, se deserializa **model.joblib**.
- **CSV de salida:** se escribe solo después de validar e inferir todas las filas, en un fichero temporal que se publica al final.
- **Mensaje de error:** indica el campo o el fichero, la regla incumplida y, si es una fila, su número y su `sample_id`; el CLI lo escribe en `stderr` y termina con código 2.
- **Segunda fila inválida tras una primera correcta:** no se publica nada. Una salida parcial parece completa para quien la consume, así que se corrige la fila y se vuelve a ejecutar.

### Invariantes que debe comprobar el cargador

1. `feature_names` es exactamente `FEATURE_NAMES`: mismos nombres y mismo orden.
2. `preprocessing_version` coincide con la del código y no se confunde con `model_version`.
3. `output_labels` es `needs_review`, `acceptable`, `excellent`, y la etiqueta que devuelve el modelo debe estar en esa lista.
4. `schema_version` es `wine-quality-bundle-v1` y el manifiesto no trae claves desconocidas.
5. El estimador ofrece `predict` y `predict_proba`.

El manifiesto no garantiza la seguridad del binario: **joblib** usa `pickle`, así que solo se carga un bundle de una fuente de confianza.